In [166]:
import os
import sys
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent  
from langgraph.checkpoint.memory import MemorySaver  # 用于多轮对话

# 导入自定义工具
from tools.weather import get_weather
from tools.calculator import calculator


# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
model = init_chat_model("groq:qwen/qwen3-32b", api_key=GROQ_API_KEY)

In [25]:
print(get_weather)

name='get_weather' description='获取指定城市的天气信息\n\n参数:\n    city: 城市名称，如"北京"、"上海"\n\n返回:\n    天气信息字符串' args_schema=<class 'langchain_core.utils.pydantic.get_weather'> func=<function get_weather at 0x111c22e80>


In [26]:
agent = create_agent(
    model=model,
    tools=[get_weather],  # 只给一个工具
    system_prompt="你是一个有帮助的助手，可以查询天气信息。"
)


# 测试：需要工具的问题
response = agent.invoke({
    "messages": [{"role": "user", "content": "北京今天天气怎么样？"}]
})

print(response['messages'][-1].content)



# 测试：不需要工具的问题
response = agent.invoke({
    "messages": [{"role": "user", "content": "你好，介绍一下你自己"}]
})

print(response['messages'][-1].content)


今天北京的天气是晴天，温度大约是15°C，空气质量良好。
我是一个有帮助的助手，可以查询天气信息、回答问题、提供信息等。如果你需要查询天气信息，可以告诉我你想查询哪个城市的天气。


In [106]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculator],
    system_prompt="你是一个有帮助的助手。"
)


# 测试不同类型的问题
tests = [
    "上海的天气怎么样？",           # 应该用 get_weather
    "15 乘以 23 等于多少？",         # 应该用 calculator
    "我是谁？"
]


In [83]:
response=agent.invoke({
    "messages": [{"role": "user", "content": "上海天气怎么样？"}]
})
print(len(response))
# print(response)

1


In [102]:
print(response)
print(response.get("messages"))
print(response["messages"])
print(response["messages"][0])
print(response["messages"][1])
print("---------")
print(response["messages"][1].tool_calls)
print("---------")
print(response["messages"][2])
print(response["messages"][3])
print(len(response["messages"]))
print(response["messages"][3].content)
print(response["messages"][3].response_metadata)

{'messages': [HumanMessage(content='上海天气怎么样？', additional_kwargs={}, response_metadata={}, id='40557689-1da8-4e89-a799-4542358ac9a1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zfbt5867n', 'function': {'arguments': '{"city":"上海"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 459, 'total_tokens': 473, 'completion_time': 0.045748692, 'completion_tokens_details': None, 'prompt_time': 0.076382387, 'prompt_tokens_details': None, 'queue_time': 0.048376857, 'total_time': 0.122131079}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d43e1-c991-7ec2-8be0-340f982de721-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'zfbt5867n', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 459, 'output_tokens'

In [103]:
for i, question in enumerate(tests, 1):
    print(f"测试 {i}：{question}")
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    # 显示最终回答
    # print(f"Agent 回复：{response['messages'][-1].content}")
    # print(response)
    print(len(response['messages']))
    print(response['messages'][1])
    # print(response['messages'].tool_calls)


测试 1：上海的天气怎么样？
4
content='' additional_kwargs={'tool_calls': [{'id': 'xae014ex1', 'function': {'arguments': '{"city":"上海"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 460, 'total_tokens': 474, 'completion_time': 0.036977723, 'completion_tokens_details': None, 'prompt_time': 0.023412122, 'prompt_tokens_details': None, 'queue_time': 0.099250805, 'total_time': 0.060389845}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d43e7-42db-7d23-83f5-67cb7f504a87-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'xae014ex1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 460, 'output_tokens': 14, 'total_tokens': 474}
测试 2：15 乘以 23 等于多少？
4
content='' additional_kwargs={'tool_calls': [{'id': 'zmx51m4k0', 'function': {'argume

In [107]:
for question in tests:
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    # 显示最终回答
    print(f"Agent 回复：{response['messages'][-1].content}")

Agent 回复：上海的天气是多云，温度18°C，有轻微雾霾。
Agent 回复：结果是345。
Agent 回复：您好，我是一名助手，我的目的是帮助和协助您解决问题和回答问题。如果您有任何具体的问题或需要帮助，请随时问我，我将尽力提供您需要的信息和帮助。


In [109]:
system_message = """你是一个友好的助手。
特点：
- 回答简洁明了
- 使用工具前先说明
- 结果用表格或列表清晰展示"""

agent = create_agent(
    model=model,
    tools=[get_weather, calculator],
    system_prompt=system_message  
)


response = agent.invoke({
    "messages": [{"role": "user", "content": "北京天气如何？顺便算一下 100 加 50"}]
})

print(f"Agent 回复：{response['messages'][-1].content}")



Agent 回复：北京的天气是晴天，温度为15°C，空气质量良好。另外，100加50的结果是150。


In [110]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "北京天气如何？顺便算一下 100 加 50"}]
})


In [147]:
len(response['messages'])
for i in range(len(response['messages'])):
    print(f"{i}:  {response['messages'][i]}")

print("使用工具数量： ",len(response["messages"][1].tool_calls))

print(response["messages"][1].tool_calls[0])

for i in range(len(response["messages"][1].tool_calls)):
    print("使用工具: ", response["messages"][1].tool_calls[i].get("name"))

print("\n--------------------------")
for i in range(len(response["messages"][1].tool_calls)):
    print(response["messages"][1].tool_calls[i])



print("\n--------------------------")
print(response["messages"][1].tool_calls)

0:  content='北京天气如何？顺便算一下 100 加 50' additional_kwargs={} response_metadata={} id='43049adb-2344-4ad6-8975-f1445d9df3b7'
1:  content='要了解北京的天气信息，我将使用“获取指定城市的天气信息”工具。要计算100加50，我将使用“执行基本的数学计算”工具。\n\n' additional_kwargs={'tool_calls': [{'id': 'fmy8qv2cv', 'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'tbmkyw0dx', 'function': {'arguments': '{"a":100,"b":50,"operation":"add"}', 'name': 'calculator'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 496, 'total_tokens': 572, 'completion_time': 0.220932163, 'completion_tokens_details': None, 'prompt_time': 0.053749588, 'prompt_tokens_details': None, 'queue_time': 0.105031707, 'total_time': 0.274681751}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d43e9-a829-75b0-a70b-cfb81b72954a-0' tool_calls=

In [153]:
agent = create_agent(
    model=model,
    tools=[calculator],
system_prompt="你是一个有帮助的助手。"
)



response = agent.invoke({
    "messages": [{"role": "user", "content": "25 乘以 8 等于多少？"}]
})


for i, msg in enumerate(response['messages'], 1):
    print(f"\n--- 消息 {i} ({msg.__class__.__name__}) ---")
    if hasattr(msg, 'content'):
        print(f"内容：{msg.content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用：{msg.tool_calls}")



--- 消息 1 (HumanMessage) ---
内容：25 乘以 8 等于多少？

--- 消息 2 (AIMessage) ---
内容：
工具调用：[{'name': 'calculator', 'args': {'a': 25, 'b': 8, 'operation': 'multiply'}, 'id': '4dqyxtkaz', 'type': 'tool_call'}]

--- 消息 3 (ToolMessage) ---
内容：25.0 multiply 8.0 = 200.0

--- 消息 4 (AIMessage) ---
内容：200


In [152]:
agent = create_agent(
    model=model,
    tools=[calculator,get_weather],
system_prompt="你是一个有帮助的助手。"
)



response = agent.invoke({
    "messages": [{"role": "user", "content": "25 乘以 8 等于多少？以及北京的天气怎样？"}]
})


for i, msg in enumerate(response['messages'], 1):
    print(f"\n--- 消息 {i} ({msg.__class__.__name__}) ---")
    if hasattr(msg, 'content'):
        print(f"内容：{msg.content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用：{msg.tool_calls}")



--- 消息 1 (HumanMessage) ---
内容：25 乘以 8 等于多少？以及北京的天气怎样？

--- 消息 2 (AIMessage) ---
内容：
工具调用：[{'name': 'calculator', 'args': {'a': 25, 'b': 8, 'operation': 'multiply'}, 'id': '914h3069r', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'zk89j9ykn', 'type': 'tool_call'}]

--- 消息 3 (ToolMessage) ---
内容：25.0 multiply 8.0 = 200.0

--- 消息 4 (ToolMessage) ---
内容：晴天，温度 15°C，空气质量良好

--- 消息 5 (AIMessage) ---
内容：25 乘以 8 等于 200。北京的天气是晴天，温度 15°C，空气质量良好。


In [168]:
memory = MemorySaver()

# 创建带记忆的 Agent
agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="你是一个有帮助的助手。",
    checkpointer=memory  # ✅ 添加检查点以支持多轮对话
)

# 使用 thread_id 来保持对话
config = {"configurable": {"thread_id": "conversation-1"}}

# 第一轮
print("\n用户：10 加 5 等于多少？")
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "10 加 5 等于多少？"}]},
    config=config
)
print(f"Agent：{response1['messages'][-1].content}")

# 第二轮：继续上一轮的对话（记忆自动保持）
print("\n用户：再乘以 3 呢？")
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "再乘以 3 呢？"}]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent：{response2['messages'][-1].content}")




用户：10 加 5 等于多少？
Agent：10 加 5 的结果是 **15**。

用户：再乘以 3 呢？
Agent：15 乘以 3 的结果是 **45**。


In [174]:
agent = create_agent(
    model=model,
    tools=[calculator,get_weather],
system_prompt="你是一个有帮助的助手。"
)

# 第一轮
response1 = agent.invoke({
    "messages": [{"role": "user", "content": "10 + 5"}]
})

# for i, msg in enumerate(response1['messages'], 1):
#     print(f"\n--- 消息 {i} ({msg.__class__.__name__}) ---")
#     if hasattr(msg, 'content'):
#         print(f"内容：{msg.content}")
#     if hasattr(msg, 'tool_calls') and msg.tool_calls:
#         print(f"工具调用：{msg.tool_calls}")


# print(response1['messages'])

# 第二轮（带历史）
response2 = agent.invoke({
    "messages": response1['messages'] + [
        {"role": "user", "content": "再乘以 3"}
    ]
})
print("len(response2['messages']):  ",len(response2['messages']))
for i, msg in enumerate(response2['messages'], 1):
    print(f"\n--- 消息 {i} ({msg.__class__.__name__}) ---")
    if hasattr(msg, 'content'):
        print(f"内容：{msg.content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用：{msg.tool_calls}")

len(response2['messages']):   8

--- 消息 1 (HumanMessage) ---
内容：10 + 5

--- 消息 2 (AIMessage) ---
内容：
工具调用：[{'name': 'calculator', 'args': {'a': 10, 'b': 5, 'operation': 'add'}, 'id': '3p7tdqyg9', 'type': 'tool_call'}]

--- 消息 3 (ToolMessage) ---
内容：10.0 add 5.0 = 15.0

--- 消息 4 (AIMessage) ---
内容：10 + 5 = 15

--- 消息 5 (HumanMessage) ---
内容：再乘以 3

--- 消息 6 (AIMessage) ---
内容：
工具调用：[{'name': 'calculator', 'args': {'a': 15, 'b': 3, 'operation': 'multiply'}, 'id': '8pr160tdv', 'type': 'tool_call'}]

--- 消息 7 (ToolMessage) ---
内容：15.0 multiply 3.0 = 45.0

--- 消息 8 (AIMessage) ---
内容：15 × 3 = 45
